# Day 2 — Streaming and Human-in-the-Loop

**Time:** 75–90 minutes  
**Build:** stream a bounded debugging loop's progress, then let a person decide at each step whether it continues.  
**Focus:** streaming changes *when* a caller sees progress; a human-in-the-loop changes *who* decides whether the loop continues.

Day 1 bounded a loop with deterministic code. Today we add two things to that loop:

1. **Streaming** — surface each step as its node finishes, instead of waiting for the final state.
2. **Human-in-the-loop** — a `human_review` node pauses the graph with `interrupt()` after every step, and the caller resumes it with `Command(resume=...)`. The iteration limit still stops the loop even if every answer is `retry`.

A pause must survive between two separate `invoke` calls, so the human-in-the-loop graph uses a checkpointer keyed by a `thread_id`.

Before running the notebook, complete the shared setup in the repository's main `README.md` and start MLflow.

## 1. Connect to the shared MLflow Gateway

In [1]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="http://127.0.0.1:5001/gateway/mlflow/v1",
    api_key="not-needed",
    model="workshop-gemini",
)

## 2. Define the running example and shared imports

Day 1 imported these types while building its graphs. This notebook stands on its own, so we import the shared types once and restate the running knapsack bug here.

In [ ]:
from typing import Literal, TypedDict
from pydantic import BaseModel
from langgraph.graph import END, START, StateGraph
import sys
sys.path.append("../..")
from graph_image import show_graph

problem_statement = "Choose items within the capacity and maximize value. Each item may be selected at most once."

buggy_code = """
n, capacity = map(int, input().split())
items = [tuple(map(int, input().split())) for _ in range(n)]
dp = [0] * (capacity + 1)
for weight, value in items:
    for current in range(weight, capacity + 1):
        dp[current] = max(dp[current], dp[current - weight] + value)
print(dp[capacity])
"""

## 3. Define the debug step

Both parts below reuse this step. The schema separates thought, action, observation, diagnosis, fix, and a `continue_loop` flag, so each step is inspectable — a streamed run can print it, and a human reviewer can judge it.

In [ ]:
DEBUG_PROMPT = """
You are a Python debugging agent. Inspect one bug per turn.

For each turn:
- Explain your thought.
- State the inspection action you took.
- Report the exact evidence you observed.
- Give a diagnosis and a fix.

Set continue_loop to true only when another bug may remain.
Do not repeat a bug already found in the previous steps.
Do not invent evidence that is not present in the code.
"""

class DebugStep(BaseModel):
    thought: str
    action: str
    observation: str
    diagnosis: str
    fix: str
    continue_loop: bool

step_llm = llm.with_structured_output(DebugStep)

# Part 1 — Streaming Progress

`invoke()` returns only the final state, after every node has run. A multi-step loop can take many seconds, and a person watching learns nothing until it finishes.

`stream()` yields an ordered update as **each node finishes**, so a caller can show progress immediately. The same graph supports several stream modes:

- `updates` — the state change each node returns (used below).
- `values` — the full state after each step.
- `messages` — token-by-token model output.

This is the in-process version of a progress stream. Day 7 puts the same idea behind an HTTP endpoint as **Server-Sent Events (SSE)**, so a browser can render each step as it arrives.

## 4. Build a streaming loop

We put the debug step in a self-loop bounded by `limit`, so the whole run streams to completion. The router is deterministic: keep looping while the last step asks to continue and the limit is not reached.

In [ ]:
class StreamState(TypedDict):
    problem: str
    code: str
    iteration: int
    limit: int
    steps: list[dict]

def stream_analyze(state: StreamState):
    step = step_llm.invoke([
        {"role": "system", "content": DEBUG_PROMPT},
        {"role": "user", "content": f"Problem:\n{state['problem']}\n\nCode:\n{state['code']}\n\nPrevious steps:\n{state['steps']}"},
    ])
    return {
        "iteration": state["iteration"] + 1,
        "steps": state["steps"] + [step.model_dump()],
    }

def stream_route(state: StreamState) -> Literal["stream_analyze", "end"]:
    keep_going = state["steps"][-1]["continue_loop"]
    return "stream_analyze" if keep_going and state["iteration"] < state["limit"] else "end"

builder = StateGraph(StreamState)
builder.add_node("stream_analyze", stream_analyze)
builder.add_edge(START, "stream_analyze")
builder.add_conditional_edges("stream_analyze", stream_route, {"stream_analyze": "stream_analyze", "end": END})
stream_graph = builder.compile()
show_graph(stream_graph)

## 5. Stream ordered progress updates

`stream_mode="updates"` yields one chunk per node as it completes, shaped as `{node_name: state_update}`. Read each chunk the moment it arrives instead of waiting for the final state.

In [ ]:
final = None
for chunk in stream_graph.stream(
    {"problem": problem_statement, "code": buggy_code, "iteration": 0, "limit": 5, "steps": []},
    stream_mode="updates",
):
    node, update = next(iter(chunk.items()))
    latest = update["steps"][-1]
    print(f"[{node}] step {update['iteration']}: {latest['diagnosis']}")
    final = update

assert final is not None and 1 <= final["iteration"] <= 5
print("\nStream finished after", final["iteration"], "step(s).")

## Break, inspect, reflect

Change `stream_mode` to `"values"` and print `chunk["iteration"]` each step. Notice that `updates` shows only what a node changed, while `values` repeats the full state every step. Then run the same graph through `invoke()` and compare what you see.

**Exit question:** Streaming lets a caller display progress as it happens. Does streaming change what the graph computes, or only when the caller sees it?

# Part 2 — Human-in-the-Loop Review

The streaming loop ran to completion on its own. Now a person reviews each step and decides whether the loop may try again or must stop. LangGraph pauses the graph with `interrupt()` and resumes it later from that exact point, so the pause can last as long as the human needs.

We keep the same debug step and add one new node, `human_review`, that pauses after every step.

## 6. Add a `human_review` node that pauses with `interrupt()`

`interrupt()` stops the node and sends a JSON-serializable payload out of the graph — here, the latest diagnosis and fix. The graph does not resume on its own; the caller must invoke it again with `Command(resume=...)`, and that resume value becomes the return value of `interrupt()` inside the node.

A pause has to survive between two separate `invoke` calls, so the graph needs a checkpointer. `MemorySaver` keeps that state in memory, keyed by a `thread_id` that every `invoke` call for this run must share. Because the checkpointer serializes every field, `steps` stores plain dicts (`step.model_dump()`), not `DebugStep` objects.

`human_review` makes its routing decision with `Command(goto=...)` instead of `add_conditional_edges`. Nothing happens in the node before the `interrupt()` call — if a model call ran first, it would run again every time the node resumes.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

class HumanLoopState(TypedDict):
    problem: str
    code: str
    iteration: int
    limit: int
    steps: list[dict]

def hl_analyze(state: HumanLoopState):
    step = step_llm.invoke([
        {"role": "system", "content": DEBUG_PROMPT},
        {"role": "user", "content": f"Problem:\n{state['problem']}\n\nCode:\n{state['code']}\n\nPrevious steps:\n{state['steps']}"},
    ])
    return {
        "iteration": state["iteration"] + 1,
        "steps": state["steps"] + [step.model_dump()],
    }

def human_review(state: HumanLoopState) -> Command[Literal["hl_analyze", "__end__"]]:
    latest = state["steps"][-1]
    decision = interrupt({
        "question": "Try again or end?",
        "iteration": state["iteration"],
        "diagnosis": latest["diagnosis"],
        "fix": latest["fix"],
        "continue_loop": latest["continue_loop"],
    })

    if decision == "retry" and state["iteration"] < state["limit"]:
        return Command(goto="hl_analyze")
    return Command(goto=END)

builder = StateGraph(HumanLoopState)
builder.add_node("hl_analyze", hl_analyze)
builder.add_node("human_review", human_review)
builder.add_edge(START, "hl_analyze")
builder.add_edge("hl_analyze", "human_review")

hl_graph = builder.compile(checkpointer=MemorySaver())
show_graph(hl_graph)

## 7. Run the loop and decide at each pause

`invoke` returns as soon as the graph pauses. LangGraph reports the pause as a `__interrupt__` key holding the payload passed to `interrupt()`; that key is absent once the graph reaches `END`. Read the diagnosis, type a real decision, and resume with `Command(resume=decision)` using the same `thread_id`.

Before running, predict how many times you will be asked, and what happens if you keep answering `retry` past the fifth step.

In [ ]:
import json

thread_config = {"configurable": {"thread_id": "day-2-human-loop"}}

result = hl_graph.invoke({
    "problem": problem_statement,
    "code": buggy_code,
    "iteration": 0,
    "limit": 5,
    "steps": [],
}, config=thread_config)

while "__interrupt__" in result:
    pause = result["__interrupt__"][0].value
    print(f"\n--- Step {pause['iteration']} ---")
    print("Diagnosis:", pause["diagnosis"])
    print("Fix:", pause["fix"])

    answer = input("Type 'retry' to let the agent try again, anything else to end: ").strip().lower()
    result = hl_graph.invoke(Command(resume=answer), config=thread_config)

assert 1 <= result["iteration"] <= 5
print("\nStopped after", result["iteration"], "step(s).")
print(json.dumps(result["steps"][-1], indent=2))

## Break, inspect, reflect

Answer `retry` five times in a row. Confirm the graph still stops at `limit`, even though nothing you typed asked it to. Then remove the checkpointer from `compile()` and try resuming again.

**Exit question:** `interrupt()` re-runs its node from the first line every time that node resumes. Why does `human_review` place the `interrupt()` call before any other work, and what would go wrong if `hl_analyze` tried to pause the same way?